In [1]:
import warnings
warnings.filterwarnings("ignore")

import copy
import os
import time

import crepe
import ddsp
import ddsp.training
from ddsp.training.postprocessing import (
    detect_notes, fit_quantile_transform
)
from ddsp.training import (data, decoders, encoders, models, preprocessing, 
                           train_util, trainers)
import gin
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pickle
import tensorflow.compat.v2 as tf
import tensorflow_datasets as tfds
import soundfile as sf
import scipy.linalg

print('Done!')

2025-05-11 02:05:31.730095: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-11 02:05:32.139808: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-05-11 02:05:32.139864: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2025-05-11 02:05:33.590955: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2025-

Done!


In [2]:
# Plot spectrograms
def plot_spectrogram(waveform, title):
    D = librosa.amplitude_to_db(np.abs(librosa.stft(waveform, n_fft=1024)), ref=np.max)
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(D, sr=16000, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [3]:
# Step 1: Load the full NSynth TFDS dataset (train split)
provider = data.NSynthTfds(split='train')
dataset = provider.get_dataset(shuffle=False)

# Step 2: Filter for instrument_family == 2 (Guitar)
def filter_guitar(example):
    return tf.equal(example['instrument_family'], 2)

filtered_dataset = dataset.filter(filter_guitar)

# Step 3: Batch and prefetch
batched_dataset = filtered_dataset.take(1).batch(1).repeat().prefetch(tf.data.AUTOTUNE)

# Step 4: Get a batch of 10 examples
batch = next(iter(batched_dataset))
audio = batch['audio']
n_samples = audio.shape[1]

# Optional: print keys and shapes
for k, v in batch.items():
    print(f"{k}: {v.shape}")
    
print(f"Number of Samples: {n_samples}")

2025-05-11 02:08:50.893373: W tensorflow/core/platform/cloud/google_auth_provider.cc:184] All attempts to get a Google authentication bearer token failed, returning an empty token. Retrieving token from files failed with "NOT_FOUND: Could not locate the credentials file.". Retrieving token from GCE failed with "FAILED_PRECONDITION: Error executing an HTTP request: libcurl code 6 meaning 'Couldn't resolve host name', error details: Could not resolve host: metadata".
2025-05-11 02:08:53.559625: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2025-05-11 02:08:53.559682: W tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:265] failed call to cuInit: UNKNOWN ERROR (303)
2025-05-11 02:08:53.559716: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (4

audio: (1, 64000)
f0_hz: (1, 1000)
f0_confidence: (1, 1000)
loudness_db: (1, 1000)
pitch: (1,)
instrument_source: (1,)
instrument_family: (1,)
instrument: (1,)
Number of Samples: 64000
